# 08 RL Grasping Gridworld

RL 用 state、action、reward、transition 和 policy 表达交互学习。

![RL gridworld visual map](assets/figures/08_rl_gridworld.png)

## Learning Objectives

- Map a manipulation problem into state, action, reward, transition, and policy.
- Explain why reward design changes what the agent learns.
- Use a tiny gridworld to reason about exploration before moving to robot simulators.

## Checkpoint

- Run the trajectory cell and identify the state after each action.
- Explain what event produces the positive reward.
- Describe how the shortest action sequence changes when the object moves.

## Practice Task

Move the object to a new grid cell and write a shortest successful action sequence before running it.

## Concept Map

![Colab concept image](assets/colab/08_rl_gridworld_concept.png)

**Concept image.** A tiny gridworld makes state, action, transition, reward, and policy choices inspectable.

In [ ]:
from pathlib import Path
import sys

COLAB_REPO_URL = "https://github.com/Hollis36/robotic-manipulation-learning.git"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    in_colab = "google.colab" in sys.modules
    if in_colab:
        import subprocess

        PROJECT_ROOT = Path("/content/robotic-manipulation-learning")
        if not PROJECT_ROOT.exists():
            # Equivalent command: git clone --depth 1 <repo> <target>
            subprocess.run(["git", "clone", "--depth", "1", COLAB_REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


In [ ]:
from rml.gridworld import GridGraspWorld

env = GridGraspWorld(width=4, height=4, object_pos=(1, 0), start_pos=(0, 0))
state = env.reset()
trajectory = [("reset", state, 0.0, False)]
for action in ["right", "grasp"]:
    state, reward, done, info = env.step(action)
    trajectory.append((action, state, reward, done, info["event"]))
trajectory

## Result Figure

Draw the grid, executed trajectory, object position, and final reward event.

The figure below is generated from the values computed in this notebook. Treat it as evidence from the code, not as a decorative illustration.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
fig, ax = plt.subplots(figsize=(4.8, 4.8))
for x in range(env.width + 1):
    ax.axvline(x - 0.5, color='0.75', lw=1)
for y in range(env.height + 1):
    ax.axhline(y - 0.5, color='0.75', lw=1)
positions = [entry[1].agent_pos for entry in trajectory]
for start_cell, end_cell in zip(positions, positions[1:]):
    if start_cell != end_cell:
        ax.annotate('', xy=end_cell, xytext=start_cell, arrowprops={'arrowstyle': '->', 'lw': 2})
ax.scatter([env.start_pos[0]], [env.start_pos[1]], marker='s', s=150, label='start')
ax.scatter([env.object_pos[0]], [env.object_pos[1]], marker='*', s=220, label='object')
ax.scatter([positions[-1][0]], [positions[-1][1]], marker='x', s=160, label='final agent')
final_event = trajectory[-1][-1]
final_reward = trajectory[-1][2]
ax.set_title(f'final reward={final_reward:.1f}, event={final_event}')
ax.set_xlim(-0.5, env.width - 0.5)
ax.set_ylim(-0.5, env.height - 0.5)
ax.set_xticks(range(env.width))
ax.set_yticks(range(env.height))
ax.set_aspect('equal', adjustable='box')
ax.legend(frameon=False, loc='upper left')
plt.show()

## Parameter Experiment

The next cell is marked with `COLAB_PARAMETER_EXPERIMENT` so it is easy to find in Colab. Move the object and print the outcome of a shortest handwritten action sequence.

In [ ]:
# COLAB_PARAMETER_EXPERIMENT
def shortest_actions(start_pos, object_pos):
    actions = []
    dx = object_pos[0] - start_pos[0]
    dy = object_pos[1] - start_pos[1]
    actions.extend(['right'] * max(0, dx))
    actions.extend(['left'] * max(0, -dx))
    actions.extend(['up'] * max(0, dy))
    actions.extend(['down'] * max(0, -dy))
    return actions + ['grasp']

for object_pos in [(1, 0), (2, 1), (3, 3)]:
    trial_env = GridGraspWorld(width=4, height=4, object_pos=object_pos, start_pos=(0, 0))
    state = trial_env.reset()
    total_reward = 0.0
    event = 'reset'
    actions = shortest_actions(trial_env.start_pos, object_pos)
    for action in actions:
        state, reward, done, info = trial_env.step(action)
        total_reward += reward
        event = info['event']
        if done:
            break
    print('object_pos=', object_pos, 'actions=', actions, 'event=', event, 'total_reward=', round(total_reward, 2), 'holding=', state.holding_object)

## Reflection Prompt

为什么同一个 reward 规则在 object_pos 改变后仍然能评价成功？说明 state、action sequence 和 terminal reward 的关系。

Exercise: 改变 object_pos，写出新的最短 action sequence。